# MalApp 模型甲/乙参数高效微调与强化学习容量测算

日期：2026-07-27。目的：把 GPU 选择、训练时间、租用时长和费用估算变成可复算的规划模型。所有结果均为规划区间，正式租卡后应先跑 200 step 基准测试校准吞吐。

In [1]:
from pprint import pprint

models = {
    '模型乙': {'base': 'Qwen3-14B', 'params_b': 14.8, 'architecture': 'Dense'},
    '模型甲': {'base': 'Qwen3.6-35B-A3B-FP8', 'params_b': 35.0, 'active_b': 3.0, 'architecture': 'MoE'},
}
local_hardware = {'gpu': 'RTX 5060 Laptop', 'vram_gb': 8, 'ram_gb': 32}
print('当前模型与本地硬件：')
pprint(models)
pprint(local_hardware)


当前模型与本地硬件：
{'模型乙': {'architecture': 'Dense', 'base': 'Qwen3-14B', 'params_b': 14.8},
 '模型甲': {'active_b': 3.0,
         'architecture': 'MoE',
         'base': 'Qwen3.6-35B-A3B-FP8',
         'params_b': 35.0}}
{'gpu': 'RTX 5060 Laptop', 'ram_gb': 32, 'vram_gb': 8}


In [2]:
gpu_price_usd_h = {
    'RTX 4090 24GB': 0.69,
    'RTX A6000 48GB': 0.53,
    'A100 PCIe 80GB': 1.39,
    'H100 PCIe 80GB': 2.89,
    'RTX Pro 6000 96GB': 1.99,
    'H200 141GB': 4.39,
}
print('示例按小时价格（USD，RunPod Community Cloud 页面，2026-07-27 查阅）：')
pprint(gpu_price_usd_h)


示例按小时价格（USD，RunPod Community Cloud 页面，2026-07-27 查阅）：
{'A100 PCIe 80GB': 1.39,
 'H100 PCIe 80GB': 2.89,
 'H200 141GB': 4.39,
 'RTX 4090 24GB': 0.69,
 'RTX A6000 48GB': 0.53,
 'RTX Pro 6000 96GB': 1.99}


In [3]:
phases = [
    {'model': '模型乙', 'method': 'QLoRA SFT', 'gpu': 'A100 PCIe 80GB', 'train_min_h': 6, 'train_max_h': 12},
    {'model': '模型乙', 'method': 'DPO', 'gpu': 'A100 PCIe 80GB', 'train_min_h': 8, 'train_max_h': 18},
    {'model': '模型乙', 'method': 'GRPO', 'gpu': 'A100 PCIe 80GB', 'train_min_h': 18, 'train_max_h': 48},
    {'model': '模型甲', 'method': 'QLoRA SFT', 'gpu': 'H100 PCIe 80GB', 'train_min_h': 12, 'train_max_h': 30},
    {'model': '模型甲', 'method': 'DPO', 'gpu': 'H100 PCIe 80GB', 'train_min_h': 18, 'train_max_h': 48},
    {'model': '模型甲', 'method': 'GRPO', 'gpu': 'H200 141GB', 'train_min_h': 24, 'train_max_h': 72},
]
for row in phases:
    row['mid_h'] = (row['train_min_h'] + row['train_max_h']) / 2
    row['core_cost_min_usd'] = round(row['train_min_h'] * gpu_price_usd_h[row['gpu']], 2)
    row['core_cost_max_usd'] = round(row['train_max_h'] * gpu_price_usd_h[row['gpu']], 2)
print('核心训练区间（未含下载、评估、故障缓冲）：')
for row in phases:
    print(row)


核心训练区间（未含下载、评估、故障缓冲）：
{'model': '模型乙', 'method': 'QLoRA SFT', 'gpu': 'A100 PCIe 80GB', 'train_min_h': 6, 'train_max_h': 12, 'mid_h': 9.0, 'core_cost_min_usd': 8.34, 'core_cost_max_usd': 16.68}
{'model': '模型乙', 'method': 'DPO', 'gpu': 'A100 PCIe 80GB', 'train_min_h': 8, 'train_max_h': 18, 'mid_h': 13.0, 'core_cost_min_usd': 11.12, 'core_cost_max_usd': 25.02}
{'model': '模型乙', 'method': 'GRPO', 'gpu': 'A100 PCIe 80GB', 'train_min_h': 18, 'train_max_h': 48, 'mid_h': 33.0, 'core_cost_min_usd': 25.02, 'core_cost_max_usd': 66.72}
{'model': '模型甲', 'method': 'QLoRA SFT', 'gpu': 'H100 PCIe 80GB', 'train_min_h': 12, 'train_max_h': 30, 'mid_h': 21.0, 'core_cost_min_usd': 34.68, 'core_cost_max_usd': 86.7}
{'model': '模型甲', 'method': 'DPO', 'gpu': 'H100 PCIe 80GB', 'train_min_h': 18, 'train_max_h': 48, 'mid_h': 33.0, 'core_cost_min_usd': 52.02, 'core_cost_max_usd': 138.72}
{'model': '模型甲', 'method': 'GRPO', 'gpu': 'H200 141GB', 'train_min_h': 24, 'train_max_h': 72, 'mid_h': 48.0, 'core_cost_min_usd':

In [4]:
rental_scenarios = [
    {'scenario': '乙：SFT+DPO 生产首轮', 'gpu': 'A100 PCIe 80GB', 'rent_min_h': 36, 'rent_max_h': 48},
    {'scenario': '乙：SFT+DPO 更快方案', 'gpu': 'H100 PCIe 80GB', 'rent_min_h': 24, 'rent_max_h': 36},
    {'scenario': '甲：SFT+DPO 生产首轮', 'gpu': 'H100 PCIe 80GB', 'rent_min_h': 72, 'rent_max_h': 120},
    {'scenario': '甲：SFT+DPO 大显存方案', 'gpu': 'H200 141GB', 'rent_min_h': 48, 'rent_max_h': 96},
]
for row in rental_scenarios:
    price = gpu_price_usd_h[row['gpu']]
    row['rent_cost_min_usd'] = round(row['rent_min_h'] * price, 2)
    row['rent_cost_max_usd'] = round(row['rent_max_h'] * price, 2)
print('建议租用窗口（含环境、下载、评估和一次重跑缓冲）：')
for row in rental_scenarios:
    print(row)


建议租用窗口（含环境、下载、评估和一次重跑缓冲）：
{'scenario': '乙：SFT+DPO 生产首轮', 'gpu': 'A100 PCIe 80GB', 'rent_min_h': 36, 'rent_max_h': 48, 'rent_cost_min_usd': 50.04, 'rent_cost_max_usd': 66.72}
{'scenario': '乙：SFT+DPO 更快方案', 'gpu': 'H100 PCIe 80GB', 'rent_min_h': 24, 'rent_max_h': 36, 'rent_cost_min_usd': 69.36, 'rent_cost_max_usd': 104.04}
{'scenario': '甲：SFT+DPO 生产首轮', 'gpu': 'H100 PCIe 80GB', 'rent_min_h': 72, 'rent_max_h': 120, 'rent_cost_min_usd': 208.08, 'rent_cost_max_usd': 346.8}
{'scenario': '甲：SFT+DPO 大显存方案', 'gpu': 'H200 141GB', 'rent_min_h': 48, 'rent_max_h': 96, 'rent_cost_min_usd': 210.72, 'rent_cost_max_usd': 421.44}


In [5]:
assert all(x['train_min_h'] > 0 and x['train_max_h'] >= x['train_min_h'] for x in phases)
assert all(x['rent_min_h'] >= 24 and x['rent_max_h'] >= x['rent_min_h'] for x in rental_scenarios)
assert models['模型甲']['base'] != models['模型乙']['base']
print('校验通过：时间区间、价格映射和模型架构分离约束均有效。')


校验通过：时间区间、价格映射和模型架构分离约束均有效。


## 假设与限制

- SFT：约 10,000 条高质量样本、平均 1,500 token、2 epoch。
- DPO：约 3,000 组偏好对。
- GRPO：约 2,000 个 prompt，每个 4–8 个 rollout，输出约 512 token。
- 估时不是硬件 benchmark；8K 上下文通常会显著放大显存和时间，需用实际 200 step 测试替换区间。
- 费用不含存储、网络、镜像持久盘、税费和人民币汇率。
- 模型甲和乙底座架构不同，必须分别训练和部署 adapter。